**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/FMCG_Project/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f"/Volumes/fmcg/sportsbar-dp/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Base Path:  /Volumes/fmcg/sportsbar-dp/orders
Landing Path:  /Volumes/fmcg/sportsbar-dp/orders/landing/
Processed Path:  /Volumes/fmcg/sportsbar-dp/orders/processed/


## Bronze

In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

print("Total Rows: ", df.count())
display(df.limit(20))

Total Rows:  670


order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FDEC85101603,"Wednesday, December 03, 2025",789101,25891301,92.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85101603,"Wednesday, December 03, 2025",789101,25891502,232.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85101603,03/12/2025,789101,25891402,478.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85101603,"Wednesday, December 03, 2025",789101,25891201,349.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85101603,"Wednesday, December 03, 2025",789101,25891602,62.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85101603,"Wednesday, December 03, 2025",789101,88888888,484.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85101603,"Wednesday, December 03, 2025",ABC987,25891603,64.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85101603,"Wednesday, December 03, 2025",789101,25891402,478.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85301402,"Wednesday, December 03, 2025",789301,25891203,428.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899
FDEC85301402,03-12-2025,789301,25891302,48.0,2026-04-16T15:34:03.543Z,orders_2025_12_03.csv,21899


In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

### Staging table to process just the arrived incremental data

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

In [0]:
display(spark.sql(f"SELECT COUNT(*) AS total_count FROM {catalog}.{bronze_schema}.staging_{data_source}"))

total_count
670


### Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)
for file in files:
    dbutils.fs.mv(
        file.path,
        f"{processed_path}/{file.name}",
        True
    )

## Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};")
display(df_orders.limit(20))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FDEC83401502,"Tuesday, December 02, 2025",789401,25891203,256.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891502,218.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891403,280.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891201,262.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891203,256.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891403,280.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC84202603,"Tuesday, December 02, 2025",789202,25891502,218.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC84202603,2025/12/02,789202,25891403,267.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC84202603,"Tuesday, December 02, 2025",789202,25891601,69.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621
FDEC84202603,"Tuesday, December 02, 2025",789202,25891602,126.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621


**Transformations**

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
df_orders.count()

599

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-12-02|2025-12-03|
+----------+----------+



**Join with products**

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

display(df_joined.limit(20))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size,product_code
FDEC84202603,2025-12-02,789202,25891202,385.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28
FDEC83601603,2025-12-02,999999,25891403,390.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9
FDEC83601603,2025-12-02,789601,25891603,176.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa
FDEC83703502,2025-12-02,789703,25891203,328.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2
FDEC84720403,2025-12-02,999999,25891402,416.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49
FDEC85103601,2025-12-02,789103,25891302,80.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
FDEC83903303,2025-12-02,999999,25891101,500.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
FDEC84522603,2025-12-02,789522,25891603,67.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa
FDEC85203503,2025-12-02,999999,25891503,203.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379
FDEC85221603,2025-12-02,789221,25891603,175.0,2026-04-16T15:40:42.663Z,orders_2025_12_02.csv,19621,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa


In [0]:
df_joined.count()

528

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Staging table to process just the arrived incremental data

In [0]:
# staging for incremental data

df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

## Gold

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {catalog}.{silver_schema}.staging_{data_source};")

display(df_gold.limit(20))

order_id,date,customer_code,product_code,product_id,sold_quantity
FDEC84202603,2025-12-02,789202,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,25891202,385.0
FDEC83601603,2025-12-02,999999,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,25891403,390.0
FDEC83601603,2025-12-02,789601,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,25891603,176.0
FDEC83703502,2025-12-02,789703,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,25891203,328.0
FDEC84720403,2025-12-02,999999,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,25891402,416.0
FDEC85103601,2025-12-02,789103,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,25891302,80.0
FDEC83903303,2025-12-02,999999,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,25891101,500.0
FDEC84522603,2025-12-02,789522,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,25891603,67.0
FDEC85203503,2025-12-02,999999,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,25891503,203.0
FDEC85221603,2025-12-02,789221,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,25891603,175.0


In [0]:
df_gold.count()

528

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .option("mergeSchema", "true")\
        .mode("overwrite")\
        .saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Merging with Parent company

- Note: We want data for monthly level but child data is on daily level

**Incremental Load**

In [0]:
# df_child = your incremental daily rows

df_child = spark.sql(f"SELECT order_placement_date as date FROM {catalog}.{silver_schema}.staging_{data_source}")

incremental_month_df = df_child.select(
    F.trunc("date", "MM").alias("start_month")
).distinct()

incremental_month_df.show()

incremental_month_df.createOrReplaceTempView("incremental_months")

+-----------+
|start_month|
+-----------+
| 2025-12-01|
+-----------+



In [0]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM {catalog}.{gold_schema}.sb_fact_orders sbf
    INNER JOIN incremental_months m
        ON trunc(sbf.date, 'MM') = m.start_month
""")

print("Total Rows: ", monthly_table.count())
monthly_table.show(10)

Total Rows:  789
+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-12-02|0cb7b2f42657b625f...|       789202|        385.0|
|2025-12-02|77b6f538a9d0e0cf8...|       999999|        390.0|
|2025-12-02|451f7167b28a25bde...|       789601|        176.0|
|2025-12-02|889c67757ece9c973...|       789703|        328.0|
|2025-12-02|fe5a8036be4b9a787...|       999999|        416.0|
|2025-12-02|d9ebd1ca64d23951a...|       789103|         80.0|
|2025-12-02|e91ba9d665f90254d...|       999999|        500.0|
|2025-12-02|451f7167b28a25bde...|       789522|         67.0|
|2025-12-02|062f5574bbdf4386b...|       999999|        203.0|
|2025-12-02|451f7167b28a25bde...|       789221|        175.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
monthly_table.select('date').distinct().orderBy('date').show()

+----------+
|      date|
+----------+
|2025-12-01|
|2025-12-02|
|2025-12-03|
+----------+



In [0]:
df_monthly_recalc = (
    monthly_table
    .withColumn("month_start", F.trunc("date", "MM"))
    .groupBy("month_start", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start", "date")   # month_start → date = first of month
)

df_monthly_recalc.show(10, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-12-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789202       |806.0        |
|2025-12-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|999999       |1683.0       |
|2025-12-01|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|789422       |794.0        |
|2025-12-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789221       |1197.0       |
|2025-12-01|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|789902       |457.0        |
|2025-12-01|da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05|999999       |968.0        |
|2025-12-01|c68834ceaff15846bc1892c2185dc4e4f4

In [0]:
df_monthly_recalc.count()

462

In [0]:
df_monthly_recalc.createOrReplaceTempView("temp_monthly")

print("Total rows in fact_orders before merge: ", spark.sql(f"SELECT COUNT(*) FROM {catalog}.{gold_schema}.fact_orders").collect()[0][0])

spark.sql(f"""
    MERGE INTO {catalog}.{gold_schema}.fact_orders AS target
    USING temp_monthly AS source
    ON target.date = source.date AND target.product_code = source.product_code AND target.customer_code = source.customer_code
    WHEN MATCHED THEN
      UPDATE SET *
    WHEN NOT MATCHED THEN
      INSERT *
""")

print("Merge completed successfully")

print("Total rows in fact_orders after merge: ", spark.sql(f"SELECT COUNT(*) FROM {catalog}.{gold_schema}.fact_orders").collect()[0][0])

# gold_parent_delta.alias("parent_gold").merge(df_monthly_recalc.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

Total rows in fact_orders:  96577


## Cleanup

In [0]:
%sql
DROP TABLE fmcg.bronze.staging_orders;

In [0]:
%sql
DROP TABLE fmcg.silver.staging_orders;